# Reading from the silver table/s

In [0]:
silver_df = spark.table('workspace.silver.crm_customers')

## Business transformations and modeling 

In [0]:
%sql
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,
    ci.customer_id,

    ci.first_name,
    ci.last_name,
    la.country,
    ci.marital_status,
    CASE
        WHEN ci.gender <> 'n/a' THEN ci.gender
        ELSE COALESCE(ca.gender, 'n/a')
    END AS gender,
    ca.birth_date AS birthdate,
    ci.created_date AS create_date
FROM silver.crm_customers ci
LEFT JOIN silver.erp_customers ca
    ON ci.customer_id = TRY_CAST(ca.customer_id AS INT)
LEFT JOIN silver.erp_cust_location la
    ON ci.customer_id = TRY_CAST(la.customer_id AS INT)

# Writing it to the Gold Layer

In [0]:
silver_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("workspace.gold.dim_customers")

In [0]:
%sql
SELECT * FROM workspace.gold.dim_customers